![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp-workshop/blob/master/healthcare-nlp/01.2.Resume_MedicalNer_Model_Training.ipynb)

If you are using the `spark-nlp-jsl` library, please use this  [1.4.Resume_MedicalNer_Model_Training
](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/1.4.Resume_MedicalNer_Model_Training.ipynb) notebook.

# Resume MedicalNer Model Training

Steps:
- Train a new model for a few epochs.
- Load the same model and train for more epochs on the same taxnonomy, and check stats.
- Train a model already trained on a different data

## Healthcare NLP for Data Scientists Course

If you are not familiar with the components in this notebook, you can check [Healthcare NLP for Data Scientists Udemy Course](https://www.udemy.com/course/healthcare-nlp-for-data-scientists/) and the [MOOC Notebooks](https://github.com/JohnSnowLabs/spark-nlp-workshop/tree/master/Spark_NLP_Udemy_MOOC/Healthcare_NLP) for each components.

## Colab Setup

In [ ]:
# Install the johnsnowlabs library to access Spark-OCR and Spark-NLP for Healthcare, Finance, and Legal.
! pip install -q johnsnowlabs

In [ ]:
from google.colab import files
print('Please Upload your John Snow Labs License using the button below')
license_keys = files.upload()

In [ ]:
from johnsnowlabs import nlp, medical, visual

# After uploading your license run this to install all licensed Python Wheels and pre-download Jars the Spark Session JVM
nlp.settings.enforce_versions=True
nlp.install(refresh_install=True)

In [ ]:
from johnsnowlabs import nlp, medical, visual
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Automatically load license data and start a session with all jars user has access to
spark = nlp.start()

In [5]:
spark

## Download Clinical Word Embeddings for training

In [6]:
clinical_embeddings = nlp.WordEmbeddingsModel.pretrained('embeddings_clinical', "en", "clinical/models")\
    .setInputCols(["sentence", "token"])\
    .setOutputCol("embeddings")

embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[OK!]


## Download Data for Training (NCBI Disease Dataset)

In [7]:
!wget -q https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp-workshop/master/tutorials/Certification_Trainings/Healthcare/data/NCBI_disease_official_test.conll
!wget -q https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp-workshop/master/tutorials/Certification_Trainings/Healthcare/data/NCBI_disease_official_train_dev.conll

In [8]:
# from sparknlp.training import CoNLL

training_data = nlp.CoNLL().readDataset(spark, 'NCBI_disease_official_train_dev.conll')

training_data.show(3)

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|                text|            document|            sentence|               token|                 pos|               label|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|Identification of...|[{document, 0, 89...|[{document, 0, 89...|[{token, 0, 13, I...|[{pos, 0, 13, NN,...|[{named_entity, 0...|
|The adenomatous p...|[{document, 0, 21...|[{document, 0, 21...|[{token, 0, 2, Th...|[{pos, 0, 2, NN, ...|[{named_entity, 0...|
|Complex formation...|[{document, 0, 63...|[{document, 0, 63...|[{token, 0, 6, Co...|[{pos, 0, 6, NN, ...|[{named_entity, 0...|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
only showing top 3 rows



In [9]:
test_data = nlp.CoNLL().readDataset(spark, 'NCBI_disease_official_test.conll')

test_data.show(3)

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|                text|            document|            sentence|               token|                 pos|               label|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|Clustering of mis...|[{document, 0, 10...|[{document, 0, 10...|[{token, 0, 9, Cl...|[{pos, 0, 9, NN, ...|[{named_entity, 0...|
|Ataxia - telangie...|[{document, 0, 13...|[{document, 0, 13...|[{token, 0, 5, At...|[{pos, 0, 5, NN, ...|[{named_entity, 0...|
|The risk of cance...|[{document, 0, 15...|[{document, 0, 15...|[{token, 0, 2, Th...|[{pos, 0, 2, NN, ...|[{named_entity, 0...|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
only showing top 3 rows



## Split the test data into two parts:
- We Keep the first part separate and use it for training the model further, as it will be totally unseen data from the same taxonomy.

- The second part will be used to testing and evaluating

In [10]:
(test_data_1, test_data_2) = test_data.randomSplit([0.5, 0.5], seed = 100)

# save the test data as parquet for easy testing
clinical_embeddings.transform(test_data_1).write.parquet('test_1.parquet')

clinical_embeddings.transform(test_data_2).write.parquet('test_2.parquet')

## Train a new model, pause, and resume training on the same dataset.

### MedicalNerDLGraphChecker

The MedicalNerDLGraphChecker processes the dataset to extract required graph parameters (tokens, labels, embedding dimensions)

In [11]:
nerDLGraphChecker = medical.NerDLGraphChecker()\
    .setInputCols(["sentence", "token"])\
    .setLabelColumn("label")\
    .setEmbeddingsModel(clinical_embeddings)

### Train for 2 epochs

In [12]:
nerTagger = medical.NerApproach()\
    .setInputCols(["sentence", "token", "embeddings"])\
    .setLabelColumn("label")\
    .setOutputCol("ner")\
    .setMaxEpochs(2)\
    .setLr(0.003)\
    .setBatchSize(8)\
    .setRandomSeed(0)\
    .setVerbose(1)\
    .setEvaluationLogExtended(True) \
    .setEnableOutputLogs(True)\
    .setIncludeConfidence(True)\
    .setTestDataset('./test_2.parquet')\
    .setOutputLogsPath('./ner_logs')

ner_pipeline = nlp.Pipeline(stages=[
    clinical_embeddings,
    nerDLGraphChecker,
    nerTagger
 ])

In [13]:
%%time
ner_model = ner_pipeline.fit(training_data)

CPU times: user 203 ms, sys: 49.1 ms, total: 252 ms
Wall time: 9min 36s


In [14]:
!ls ner_logs/MedicalNerApproach*

ner_logs/MedicalNerApproach_7457130b149e.log


In [15]:
# Training Logs
! cat ner_logs/MedicalNerApproach*

Name of the selected graph: medical-ner-dl/blstm_10_200_128_100.pb
Training started - total epochs: 2 - lr: 0.003 - batch size: 8 - labels: 3 - chars: 84 - training examples: 6347


Epoch 1/2 started, lr: 0.003, dataset size: 6347


Epoch 1/2 - 279.33s - loss: 1935.8735 - avg training loss: 2.435061 - batches: 795
Quality on test dataset: 
time to finish evaluation: 6.71s
Total test loss: 92.1192	Avg test loss: 1.5613
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 513	 190	 23	 0.7297297	 0.95708954	 0.8280871
B-Disease	 396	 129	 68	 0.7542857	 0.8534483	 0.80080885
tp: 909 fp: 319 fn: 91 labels: 2
Macro-average	 prec: 0.74200773, rec: 0.9052689, f1: 0.8155479
Micro-average	 prec: 0.740228, rec: 0.909, f1: 0.81597847


Epoch 2/2 started, lr: 0.0029850747, dataset size: 6347


Epoch 2/2 - 257.07s - loss: 799.808 - avg training loss: 1.0060477 - batches: 795
Quality on test dataset: 
time to finish evaluation: 3.45s
Total test loss: 49.7349	Avg test loss: 0.8430
label	 tp	 fp	 fn	 prec	 re

In [16]:
# Logs of 4 consecutive epochs to compare with 2+2 epochs on separate datasets from same taxonomy

#!cat ner_logs/MedicalNerApproach_4d3d69967c3f.log

### Evaluate

In [17]:
#from sparknlp_jsl.eval import NerDLMetrics
import pyspark.sql.functions as F

pred_df = ner_model.stages[2].transform(clinical_embeddings.transform(test_data_2))

evaler = medical.NerDLMetrics(mode="full_chunk")

eval_result = evaler.computeMetricsFromDF(pred_df.select("label","ner"), prediction_col="ner", label_col="label", drop_o = True, case_sensitive = True).cache()

eval_result.withColumn("precision", F.round(eval_result["precision"],4))\
           .withColumn("recall", F.round(eval_result["recall"],4))\
           .withColumn("f1", F.round(eval_result["f1"],4)).show(100)

print(eval_result.selectExpr("avg(f1) as macro").show())
print (eval_result.selectExpr("sum(f1*total) as sumprod","sum(total) as sumtotal").selectExpr("sumprod/sumtotal as micro").show())

+-------+-----+----+----+-----+---------+------+------+
| entity|   tp|  fp|  fn|total|precision|recall|    f1|
+-------+-----+----+----+-----+---------+------+------+
|Disease|384.0|85.0|76.0|460.0|   0.8188|0.8348|0.8267|
+-------+-----+----+----+-----+---------+------+------+

+------------------+
|             macro|
+------------------+
|0.8266953713670614|
+------------------+

None
+------------------+
|             micro|
+------------------+
|0.8266953713670614|
+------------------+

None


`ner_utils`: This new module is used after NER training to calculate mertic chunkbase and plot training logs.

`evaluate`: if verbose, returns overall performance, as well as performance per chunk type; otherwise, simply returns overall precision, recall, f1 scores

`loss_plot`: Plots the figure of loss vs epochs

`get_charts` : Plots the figures of metrics ( precision, recall, f1) vs epochs

```
from sparknlp_jsl.utils.ner_utils import get_charts, loss_plot, evaluate

metrics = evaluate(preds_df['prediction'].values, preds_df['ground_truth'].values)

loss_plot(f"./ner_logs/{log_file}")

get_charts('./ner_logs/'+log_files[0])
```

### Save the model to disk

In [18]:
ner_model.stages[2].write().overwrite().save('models/NCBI_NER_2_epoch')

### Train using the saved model on unseen dataset
#### We use unseen data from same taxonomy

In [19]:
nerTagger = medical.NerApproach()\
      .setInputCols(["sentence", "token", "embeddings"])\
      .setLabelColumn("label")\
      .setOutputCol("ner")\
      .setMaxEpochs(2)\
      .setLr(0.003)\
      .setBatchSize(8)\
      .setRandomSeed(0)\
      .setVerbose(1)\
      .setEvaluationLogExtended(True) \
      .setEnableOutputLogs(True)\
      .setIncludeConfidence(True)\
      .setTestDataset('/content/test_2.parquet')\
      .setOutputLogsPath('ner_logs')\
      .setPretrainedModelPath("models/NCBI_NER_2_epoch") ## load existing model

ner_pipeline = nlp.Pipeline(stages=[
      clinical_embeddings,
      nerDLGraphChecker,
      nerTagger
 ])

In [20]:
# remove the existing logs

! rm -r ./ner_logs

In [21]:
%%time
ner_model_retrained = ner_pipeline.fit(test_data_1)

CPU times: user 85.7 ms, sys: 15.5 ms, total: 101 ms
Wall time: 1min 14s


In [22]:
!ls ner_logs/MedicalNerApproach*

ner_logs/MedicalNerApproach_852c9188ccc4.log


In [23]:
!cat ./ner_logs/MedicalNerApproach*

Name of the selected graph: pretrained graph
Training started - total epochs: 2 - lr: 0.003 - batch size: 8 - labels: 3 - chars: 84 - training examples: 477


Epoch 1/2 started, lr: 0.003, dataset size: 477


Epoch 1/2 - 33.43s - loss: 70.730606 - avg training loss: 1.1595181 - batches: 61
Quality on test dataset: 
time to finish evaluation: 4.41s
Total test loss: 43.4679	Avg test loss: 0.7367
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 483	 78	 53	 0.86096257	 0.9011194	 0.8805834
B-Disease	 417	 65	 47	 0.8651452	 0.8987069	 0.88160676
tp: 900 fp: 143 fn: 100 labels: 2
Macro-average	 prec: 0.8630539, rec: 0.8999132, f1: 0.8810983
Micro-average	 prec: 0.8628955, rec: 0.9, f1: 0.88105726


Epoch 2/2 started, lr: 0.0029850747, dataset size: 477


Epoch 2/2 - 21.92s - loss: 51.198708 - avg training loss: 0.8393231 - batches: 61
Quality on test dataset: 
time to finish evaluation: 4.42s
Total test loss: 39.0664	Avg test loss: 0.6621
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 500	 98	 36	

In [24]:
#from sparknlp_jsl.eval import NerDLMetrics
import pyspark.sql.functions as F

pred_df = ner_model_retrained.stages[2].transform(clinical_embeddings.transform(test_data_2))

evaler = medical.NerDLMetrics(mode="full_chunk")

eval_result = evaler.computeMetricsFromDF(pred_df.select("label","ner"), prediction_col="ner", label_col="label",case_sensitive = True).cache()

eval_result.withColumn("precision", F.round(eval_result["precision"],4))\
    .withColumn("recall", F.round(eval_result["recall"],4))\
    .withColumn("f1", F.round(eval_result["f1"],4)).show(100)

print(eval_result.selectExpr("avg(f1) as macro").show())
print (eval_result.selectExpr("sum(f1*total) as sumprod","sum(total) as sumtotal").selectExpr("sumprod/sumtotal as micro").show())

+-------+-----+----+----+-----+---------+------+------+
| entity|   tp|  fp|  fn|total|precision|recall|    f1|
+-------+-----+----+----+-----+---------+------+------+
|Disease|417.0|80.0|43.0|460.0|    0.839|0.9065|0.8715|
+-------+-----+----+----+-----+---------+------+------+

+------------------+
|             macro|
+------------------+
|0.8714733542319749|
+------------------+

None
+------------------+
|             micro|
+------------------+
|0.8714733542319749|
+------------------+

None


### Train a Model Choosing Best Model

`useBestModel`:  This param preserves and restores the model that has achieved the best performance at the end of the training. The priority is metrics from testDataset (micro F1), metrics from validationSplit (micro F1), and if none is set it will keep track of loss during the training. <br/>

`earlyStopping`: You can stop training at the point when the perforfmance on test/validation dataset starts to degrage. Two params are used in order to use this feature: <br/>

- `earlyStoppingCriterion`: This is used set the minimal improvement of the test metric to terminate training. The metric monitored is the same as the metrics used in `useBestModel` (micro F1 when using test/validation set, loss otherwise). Default is 0 which means no early stopping is applied.

- `earlyStoppingPatience`:  The number of epoch without improvement which will be tolerated. Default is 0, which means that early stopping will occur at the first time when performance in the current epoch is no better than in the previous epoch.

In [25]:
# remove the existing logs

! rm -r ./ner_logs

In [26]:
nerTagger = medical.NerApproach()\
      .setInputCols(["sentence", "token", "embeddings"])\
      .setLabelColumn("label")\
      .setOutputCol("ner")\
      .setMaxEpochs(20)\
      .setLr(0.003)\
      .setBatchSize(8)\
      .setRandomSeed(0)\
      .setVerbose(1)\
      .setEvaluationLogExtended(True) \
      .setEnableOutputLogs(True)\
      .setIncludeConfidence(True)\
      .setTestDataset('./test_2.parquet')\
      .setOutputLogsPath('./ner_logs')\
      .setValidationSplit(0.2)\
      .setUseBestModel(True)\
      .setEarlyStoppingCriterion(0.04)\
      .setEarlyStoppingPatience(3)


ner_pipeline = nlp.Pipeline(stages=[
      clinical_embeddings,
      nerDLGraphChecker,
      nerTagger
 ])

In [27]:
%%time
ner_model = ner_pipeline.fit(training_data)

CPU times: user 263 ms, sys: 82.4 ms, total: 346 ms
Wall time: 15min 21s


In [28]:
!ls ner_logs/MedicalNerApproach*

ner_logs/MedicalNerApproach_1c9eb1514c38.log


In [29]:
# Training Logs
! cat ner_logs/MedicalNerApproach*

Name of the selected graph: medical-ner-dl/blstm_10_200_128_100.pb
Training started - total epochs: 20 - lr: 0.003 - batch size: 8 - labels: 3 - chars: 84 - training examples: 5078


Epoch 1/20 started, lr: 0.003, dataset size: 5078


Epoch 1/20 - 232.07s - loss: 1617.5197 - avg training loss: 2.5313296 - batches: 639
Quality on validation dataset (20.0%), validation examples = 1269
time to finish evaluation: 12.49s
Total validation loss: 173.9764	Avg validation loss: 1.0942
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 1327	 205	 218	 0.866188	 0.85889965	 0.8625284
B-Disease	 989	 139	 206	 0.87677306	 0.8276151	 0.85148513
tp: 2316 fp: 344 fn: 424 labels: 2
Macro-average	 prec: 0.8714805, rec: 0.84325737, f1: 0.85713667
Micro-average	 prec: 0.8706767, rec: 0.8452555, f1: 0.85777783
Quality on test dataset: 
time to finish evaluation: 4.07s
Total test loss: 64.3600	Avg test loss: 1.0908
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 482	 101	 54	 0.82675815	 0.8992537	 0.8614835
B-Disease

As you see above, training was stopped before 20th epoch since there were not improvement.

**Evaluate**

In [30]:
#from sparknlp_jsl.eval import NerDLMetrics

pred_df = ner_model.stages[2].transform(clinical_embeddings.transform(test_data_1))

evaler = medical.NerDLMetrics(mode="full_chunk")

eval_result = evaler.computeMetricsFromDF(pred_df.select("label","ner"), prediction_col="ner", label_col="label", drop_o = True, case_sensitive = True).cache()

eval_result.withColumn("precision", F.round(eval_result["precision"],4))\
    .withColumn("recall", F.round(eval_result["recall"],4))\
    .withColumn("f1", F.round(eval_result["f1"],4)).show(100)

print(eval_result.selectExpr("avg(f1) as macro").show())
print (eval_result.selectExpr("sum(f1*total) as sumprod","sum(total) as sumtotal").selectExpr("sumprod/sumtotal as micro").show())

+-------+-----+----+----+-----+---------+------+------+
| entity|   tp|  fp|  fn|total|precision|recall|    f1|
+-------+-----+----+----+-----+---------+------+------+
|Disease|422.0|79.0|73.0|495.0|   0.8423|0.8525|0.8474|
+-------+-----+----+----+-----+---------+------+------+

+------------------+
|             macro|
+------------------+
|0.8473895582329317|
+------------------+

None
+------------------+
|             micro|
+------------------+
|0.8473895582329317|
+------------------+

None


In [ ]:
pred_df = ner_model.stages[2].transform(clinical_embeddings.transform(test_data_2))

evaler = medical.NerDLMetrics(mode="full_chunk")

eval_result = evaler.computeMetricsFromDF(pred_df.select("label","ner"), prediction_col="ner", label_col="label", drop_o = True, case_sensitive = True).cache()

eval_result.withColumn("precision", F.round(eval_result["precision"],4))\
    .withColumn("recall", F.round(eval_result["recall"],4))\
    .withColumn("f1", F.round(eval_result["f1"],4)).show(100)

print(eval_result.selectExpr("avg(f1) as macro").show())
print (eval_result.selectExpr("sum(f1*total) as sumprod","sum(total) as sumtotal").selectExpr("sumprod/sumtotal as micro").show())

## Now let's take a model trained on a different dataset and train on this dataset

In [ ]:
jsl_ner = medical.NerModel.pretrained('ner_jsl','en','clinical/models')

jsl_ner.getClasses()

### Now train a model using this model as base

In [ ]:
nerTagger = medical.NerApproach()\
      .setInputCols(["sentence", "token", "embeddings"])\
      .setLabelColumn("label")\
      .setOutputCol("ner")\
      .setMaxEpochs(2)\
      .setLr(0.003)\
      .setBatchSize(8)\
      .setRandomSeed(0)\
      .setVerbose(1)\
      .setEvaluationLogExtended(True) \
      .setEnableOutputLogs(True)\
      .setIncludeConfidence(True)\
      .setTestDataset('/content/test_2.parquet')\
      .setOutputLogsPath('ner_logs')\
      .setPretrainedModelPath("/root/cache_pretrained/ner_jsl_en_4.2.0_3.0_1666181370373")\
      .setOverrideExistingTags(True) # since the tags do not align, set this flag to true

# do hyperparameter by tuning the params above (max epoch, LR, dropout etc.) to get better results
ner_pipeline = nlp.Pipeline(stages=[
      clinical_embeddings,
      nerDLGraphChecker,
      nerTagger
 ])

In [34]:
# remove the existing logs

! rm -r ./ner_logs

In [ ]:
%%time
ner_jsl_retrained = ner_pipeline.fit(training_data)


In [36]:
!cat ./ner_logs/MedicalNerApproach*

Name of the selected graph: pretrained graph
Training started - total epochs: 2 - lr: 0.003 - batch size: 8 - labels: 3 - chars: 103 - training examples: 6347


Epoch 1/2 started, lr: 0.003, dataset size: 6347


Epoch 1/2 - 343.58s - loss: 1434.111 - avg training loss: 1.8039131 - batches: 795
Quality on test dataset: 
time to finish evaluation: 18.15s
Total test loss: 70.9840	Avg test loss: 1.2031
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 507	 139	 29	 0.78482974	 0.9458955	 0.857868
B-Disease	 415	 99	 49	 0.807393	 0.89439654	 0.8486707
tp: 922 fp: 238 fn: 78 labels: 2
Macro-average	 prec: 0.79611135, rec: 0.920146, f1: 0.85364664
Micro-average	 prec: 0.7948276, rec: 0.922, f1: 0.8537036


Epoch 2/2 started, lr: 0.0029850747, dataset size: 6347


Epoch 2/2 - 317.28s - loss: 662.8398 - avg training loss: 0.83376074 - batches: 795
Quality on test dataset: 
time to finish evaluation: 16.94s
Total test loss: 57.2999	Avg test loss: 0.9712
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 495

In [ ]:
#from sparknlp_jsl.eval import NerDLMetrics

pred_df = ner_jsl_retrained.stages[2].transform(clinical_embeddings.transform(test_data_2))

evaler = medical.NerDLMetrics(mode="full_chunk")

eval_result = evaler.computeMetricsFromDF(pred_df.select("label","ner"), prediction_col="ner", label_col="label", drop_o = True, case_sensitive = True).cache()

eval_result.withColumn("precision", F.round(eval_result["precision"],4))\
    .withColumn("recall", F.round(eval_result["recall"],4))\
    .withColumn("f1", F.round(eval_result["f1"],4)).show(100)

print(eval_result.selectExpr("avg(f1) as macro").show())
print (eval_result.selectExpr("sum(f1*total) as sumprod","sum(total) as sumtotal").selectExpr("sumprod/sumtotal as micro").show())

## Lower-Memory NER Training with `explodeSentencesForTraining` (6.4.0)

Healthcare NLP **6.4.0** adds `explodeSentencesForTraining` to `MedicalNerApproach`. When enabled, multi-sentence training rows are split into one row per sentence before training. This can reduce peak memory usage and improve training efficiency on long documents.

Validation splitting stays at the **document level** first; sentence rows are generated only for the training step. Default is `False`.

| explodeSentencesForTraining | Approx. training time | Approx. peak RAM |
|------------------------------|----------------------:|-----------------:|
| `True` | ~11 minutes | ~9.3 GB |
| `False` | ~24 minutes | ~30 GB |

Benchmark on 30 documents (~30,000 characters). Actual results vary with hardware, Spark configuration, and hyperparameters.


In [ ]:
nerTagger_explode = medical.NerApproach()\
      .setInputCols(["sentence", "token", "embeddings"])\
      .setLabelColumn("label")\
      .setOutputCol("ner")\
      .setMaxEpochs(2)\
      .setLr(0.003)\
      .setBatchSize(8)\
      .setRandomSeed(0)\
      .setVerbose(1)\
      .setValidationSplit(0.2)\
      .setEvaluationLogExtended(True) \
      .setEnableOutputLogs(True)\
      .setIncludeConfidence(True)\
      .setOutputLogsPath("./ner_logs")\
      .setExplodeSentencesForTraining(True)

explode_pipeline = nlp.Pipeline(stages=[
      clinical_embeddings,
      nerDLGraphChecker,
      nerTagger_explode,
])


In [39]:
%%time
ner_explode_model = explode_pipeline.fit(training_data)


CPU times: user 169 ms, sys: 44.7 ms, total: 214 ms
Wall time: 8min 8s
